![KAUST Academy](https://i.imgur.com/a3uAqnb.png)

# Day 44 -- Case Study: The Glitch Hunter (Starter)

The task of this morning's case study, **"The Glitch Hunter"**, from the Romanian AI olympiad's practice platform
MLCompete: competition *Algorithms to Neuron, Practice Round 1* (the **ONIA Winter Warmup 2025**). Try it yourself: the statement, the data loaded and looked at, IoU in code, and a first valid file (every pixel clean). Then write your own model; the slides show how to read the task first.

**Sources and credits.** The task, its statement and its data: [MLCompete](https://platform.olimpiada-ai.ro)
(all rights reserved by the platform), told here in our words with short quotes. The official editorial:
A.-D. Ardelean, *Editorial: The Glitch Hunter*, MLCompete, 13 December 2025 (in Romanian). A contestant's solution:
S. Asandei, [roai-solved](https://github.com/stefanasandei/roai-solved) (`contests/mlcompete/contests/winter_2025`,
MIT licence). The code of this notebook is the course's own, written for the Day 44 slides.

**How to run it.** The data is not copied here. Make a free account on the platform, open the task, download its
data, and put the folders `train/images`, `train/masks` and `test/images` in a folder `glitch` next to this notebook
(in Colab: upload them, or mount your Drive and change `DATA_DIR`). Choose a **GPU** runtime.


## The task, in our words

**The story.** You are an "Agent". A simulation is degrading: an AI model, *Project G3P1T*, paints glitches on its
images, "deliberate insertions of chaos". Your mission: find them, pixel by pixel.

**A glitch** is one of three things: a block of random digital **noise**, a rectangle with **inverted colours**, or a
compact block of **colour** that does not belong in the image.

**One data point** is one image, `train/images/img_001.png`, an array of shape (512, 512, 3). **The answer** is its
mask, `train/masks/mask_001.png`, shape (512, 512): 255 on a glitch pixel, 0 on a clean one.

**The files.** 200 training images with their masks; 200 test images without.

**What you hand in.** A ZIP of 200 PNG masks, named like the images (`img_001.png`), values 0 and 255.

**The score.** IoU (intersection over union) of your masks against the hidden ones: the pixels both mark, divided by
the pixels either marks. Higher is better. The statement does not say whether IoU is taken per image or over all
pixels; here we take the mean of the per-image IoU. (The page also shows a scoring snippet for a regression task,
`r2_score` and `pearsonr`: a leftover from another task, not this one.)


## Setup

In [ ]:
import os
import zipfile
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image

DATA_DIR = "glitch"       # holds train/images, train/masks, test/images (downloaded from the platform)
names = sorted(os.listdir(os.path.join(DATA_DIR, "train", "images")))
test_names = sorted(os.listdir(os.path.join(DATA_DIR, "test", "images")))
print(len(names), "training images,", len(test_names), "test images")

## Load

In [ ]:
images = []
masks = []
for name in names:
    images.append(np.array(Image.open(os.path.join(DATA_DIR, "train", "images", name)).convert("RGB")))
    mask_name = name.replace("img_", "mask_")
    mask = np.array(Image.open(os.path.join(DATA_DIR, "train", "masks", mask_name)).convert("L"))
    masks.append(mask > 127)                  # True on a glitch pixel
images = np.stack(images)
masks = np.stack(masks)
print(images.shape, masks.shape)

## Look: one image and its mask

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(9, 4))
axes[0].imshow(images[2])
axes[0].set_title(names[2])
shown = axes[1].imshow(masks[2], cmap="gray", vmin=0, vmax=1)
axes[1].set_title("its mask")
plt.colorbar(shown, ax=axes[1])
plt.show()
print("glitch pixels:", round(100 * masks.mean(), 1), "% of all training pixels")

## The score: IoU, and two answers with no model

In [ ]:
def iou(true_mask, pred_mask):
    both = np.logical_and(true_mask, pred_mask).sum()
    either = np.logical_or(true_mask, pred_mask).sum()
    if either == 0:
        return 1.0
    return both / either


def mean_iou(true_masks, pred_masks):
    scores = []
    for t, p in zip(true_masks, pred_masks):
        scores.append(iou(t, p))
    return np.mean(scores)


print("all clean:", mean_iou(masks, np.zeros_like(masks)))
print("all glitch:", round(mean_iou(masks, np.ones_like(masks)), 3))

## The file: a ZIP of PNG masks

In [ ]:
def write_zip(pred_masks, path="submission.zip"):
    """One PNG per test image, the image's own name, values 0 and 255."""
    os.makedirs("submission", exist_ok=True)
    with zipfile.ZipFile(path, "w") as archive:
        for name, mask in zip(test_names, pred_masks):
            Image.fromarray(mask.astype(np.uint8) * 255).save(os.path.join("submission", name))
            archive.write(os.path.join("submission", name), arcname=name)
    print(path, "holds", len(pred_masks), "masks")

In [ ]:
# A first valid file: every pixel clean. It scores 0; any model must beat it.
write_zip(np.zeros((len(test_names), 512, 512), dtype=bool))

## Your turn

Write a model that beats the two answers above, scored with `mean_iou` on validation images you keep aside.
The Day 44 slides go from here clue by clue: are all glitches rectangles, how much of an image is glitch, what is
inside a glitch, what resizing throws away, and whether a glitch has an up and a down.

Made By **Sattam Altwaim**, for the Day 44 case study; the task belongs to MLCompete.